## BLC data Ingestion Framework
NoteBook: Bronze_sql_ingestion_rearc

| Author | Date | Change Record | Description |
| ----------- | ----------- | ----------- | -----------
|Hemant kaushik	|05-Nov-2026|	-	| Notebook for ingesting BLC data
 



In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col,trim,current_timestamp,input_file_name,when,from_json
from pyspark.sql.types import StructType,StructField,StringType
import re

In [0]:
base_path = "/Volumes/workspace/hemant_poc/volume_dev/sourcefiles/"

In [0]:

bronze_config = [
    {
        "table_name": "workspace.bronze.bronze_sql_pr_series",
        "file_name": "pr.series",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_sector",
        "file_name": "pr.sector",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_measure",
        "file_name": "pr.measure",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_duration",
        "file_name": "pr.duration",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_footnote",
        "file_name": "pr.footnote",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_period",
        "file_name": "pr.period",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_seasonal",
        "file_name": "pr.seasonal",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_class",
        "file_name": "pr.class",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_data",
        "file_name": "pr.data*",
        "file_format": "csv",
        "delimiter": "\t"
    },
    {
        "table_name": "workspace.bronze.bronze_sql_pr_population",
        "file_name": "data*",
        "file_format": "json",
        "delimiter": None
    }
]

In [0]:
def clean_column_name(name):

    name = name.strip()
    name = re.sub(
        r"[\s,;{}()\n\t=]+",
        "_",
        name
    )
    return name.strip("_")


def create_bronze_sql(
    table_name,
    file_name,
    file_format,
    delimiter=None
):

    file_path = f"{base_path}/{file_name}"

    # ==========================================
    # CSV / BLS FILES
    # ==========================================

    if file_format == "csv":

        source_df = (
            spark.read
            .option("header", True)
            .option("sep", delimiter)
            .option("mode", "PERMISSIVE")
            .csv(file_path)
        )

        select_expressions = []

        for field in source_df.schema.fields:

            original_name = field.name
            clean_name = clean_column_name(original_name)

            if isinstance(field.dataType, StringType):

                select_expressions.append(
                    f"""
                    CASE
                        WHEN TRIM(`{original_name}`) = ''
                            THEN NULL
                        ELSE TRIM(`{original_name}`)
                    END AS `{clean_name}`
                    """
                )
            else:
                select_expressions.append(
                    f"`{original_name}` AS `{clean_name}`"
                )

        select_clause = ",\n".join(select_expressions)

        sql_query = f"""
            CREATE OR REPLACE TABLE {table_name}
            USING DELTA
            AS
            SELECT
                {select_clause},
                _metadata.file_path
                    AS _source_file,
                _metadata.file_modification_time
                    AS _source_modified_at,
                CURRENT_TIMESTAMP()
                    AS _ingested_at
            FROM read_files(
                '{file_path}',
                format => 'csv',
                header => true,
                sep => '{delimiter}',
                mode => 'PERMISSIVE'
            )
        """
    # ==========================================
    # JSON
    # ==========================================

    elif file_format == "json":

        sql_query = f"""
            CREATE OR REPLACE TABLE {table_name}
            USING DELTA
            AS
            SELECT
                value AS raw_json,
                _metadata.file_path
                    AS _source_file,
                _metadata.file_modification_time
                    AS _source_modified_at,
                CURRENT_TIMESTAMP()
                    AS _ingested_at
            FROM read_files(
                '{file_path}',
                format => 'text',
                wholeText => true
            )
        """
    # ==========================================
    # TEXT
        #     FROM read_files(
        #         '{file_path}',
        #         format => 'text',
        #         wholeText => true
        #     )
        # """

    else:
        raise ValueError(f"Unsupported file format: {file_format}")

    spark.sql(sql_query)
    print(f"Created: {table_name}")



In [0]:
# def create_bronze_sql(
#     table_name,
#     file_name,
#     delimitera
# ):

#     file_path = f"{base_path}/{file_name}"

#     sql_query = f"""
#         CREATE OR REPLACE TABLE {table_name}
#         USING DELTA
#         AS

#         SELECT
#             *,
#             _metadata.file_path AS _source_file,
#             _metadata.file_modification_time AS _source_modified_at,
#             current_timestamp() AS _ingested_at

#         FROM read_files(
#             '{file_path}',
#             format => 'csv',
#             header => true,
#             sep => '{delimiter}',
#             mode => 'PERMISSIVE'
#         )
#     """

#     spark.sql(sql_query)

#     print(f"Created: {table_name}")

In [0]:
for config in bronze_config:

    create_bronze_sql(
        table_name=config["table_name"],
        file_name=config["file_name"],
        file_format=config["file_format"],
        delimiter=config["delimiter"]
    )